Step 1 — Connect Google Drive and Check GPU

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

# Connect Google Drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

# Original chunking backup
CHUNK_BACKUP = Path(
    "/content/drive/MyDrive/IBM_GraphRAG/"
    "postgres_backups/ibm_chunks_overlap_v1.dump"
)

print("ENVIRONMENT CHECK")
print("=" * 45)

# Check GPU
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total",
     "--format=csv,noheader"],
    capture_output=True,
    text=True
)

if gpu.returncode == 0:
    print("✅ GPU:", gpu.stdout.strip())
else:
    print("❌ GPU not available")

# Check original chunk backup
if CHUNK_BACKUP.is_file():
    print("✅ Chunking backup found")
    print("Size:", round(CHUNK_BACKUP.stat().st_size / 1024**2, 2), "MB")
else:
    print("❌ Chunking backup not found")

print("\n✅ Step 1 completed")

ENVIRONMENT CHECK
✅ GPU: Tesla T4, 15360 MiB
✅ Chunking backup found
Size: 3.81 MB

✅ Step 1 completed


Step 2 — Install and Start Ollama

In [ ]:
import subprocess
import shutil
import time
import urllib.request
from pathlib import Path

OLLAMA_URL = "http://127.0.0.1:11434"

# 1. Install Ollama only if needed
if not shutil.which("ollama"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "curl", "zstd"],
        check=True
    )

    installer = Path("/tmp/install_ollama.sh")
    subprocess.run(
        ["curl", "-fsSL", "https://ollama.com/install.sh",
         "-o", str(installer)],
        check=True
    )
    subprocess.run(["bash", str(installer)], check=True)

print("✅ Ollama installed")

# 2. Check whether Ollama server is running
def server_ready():
    try:
        with urllib.request.urlopen(
            f"{OLLAMA_URL}/api/tags", timeout=2
        ) as response:
            return response.status == 200
    except Exception:
        return False

# 3. Start the server only if needed
if not server_ready():
    log = open("/tmp/ollama_server.log", "a")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    for _ in range(30):
        if server_ready():
            break
        time.sleep(1)

assert server_ready(), "Ollama server failed to start"

# 4. Verify
version = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True,
    check=True
)

print("✅ Ollama server running")
print("✅ API:", OLLAMA_URL)
print("✅ Version:", version.stdout.strip())

✅ Ollama installed
✅ Ollama server running
✅ API: http://127.0.0.1:11434
✅ Version: ollama version is 0.40.5


Step 3 — Download and Test Qwen Embedding Model

In [ ]:
import subprocess
import urllib.request
import json
import time

MODEL = "qwen3-embedding:0.6b"
OLLAMA_URL = "http://127.0.0.1:11434"

print("STEP 3 — QWEN EMBEDDING MODEL")
print("=" * 45)

# 1. Download the model
subprocess.run(
    ["ollama", "pull", MODEL],
    check=True
)

print("\n✅ Model downloaded")

# 2. Test embedding generation
payload = json.dumps({
    "model": MODEL,
    "input": "IBM MQ supports secure TLS connections.",
    "keep_alive": "15m"
}).encode("utf-8")

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/embed",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

start = time.perf_counter()

with urllib.request.urlopen(request, timeout=300) as response:
    result = json.load(response)

vector = result["embeddings"][0]
elapsed = time.perf_counter() - start

# 3. Verify dimensions
assert len(vector) == 1024, "Unexpected embedding dimension"

print("✅ Embedding test successful")
print("Vector dimensions:", len(vector))
print("Processing time:", round(elapsed, 2), "seconds")

# 4. Check model execution device
print("\nMODEL STATUS:")
subprocess.run(["ollama", "ps"], check=True)

print("\n✅ Step 3 completed")

STEP 3 — QWEN EMBEDDING MODEL

✅ Model downloaded
✅ Embedding test successful
Vector dimensions: 1024
Processing time: 51.21 seconds

MODEL STATUS:

✅ Step 3 completed


PostgreSQL and pgvector Setup

In [ ]:
import subprocess
import sys
import importlib.util

print("STEP 4 — POSTGRESQL SETUP")
print("=" * 45)

# 1. Install PostgreSQL and pgvector
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    [
        "apt-get", "install", "-y", "-qq",
        "postgresql",
        "postgresql-client",
        "postgresql-16-pgvector"
    ],
    check=True
)

# 2. Install Python driver if needed
if importlib.util.find_spec("psycopg") is None:
    subprocess.run(
        [sys.executable, "-m", "pip",
         "install", "-q", "psycopg[binary]"],
        check=True
    )

# 3. Start PostgreSQL
subprocess.run(
    ["service", "postgresql", "start"],
    check=True
)

# 4. Check whether our database exists
result = subprocess.run(
    [
        "runuser", "-u", "postgres", "--",
        "psql", "-At", "-d", "postgres",
        "-c",
        "SELECT EXISTS ("
        "SELECT 1 FROM pg_database "
        "WHERE datname = 'ibm_rag')"
    ],
    capture_output=True,
    text=True,
    check=True
)

database_exists = result.stdout.strip() == "t"

print("\n✅ PostgreSQL running")
print("✅ pgvector package installed")
print("✅ psycopg available")
print("Database ibm_rag exists:", database_exists)

print("\n✅ Step 4 completed")

STEP 4 — POSTGRESQL SETUP

✅ PostgreSQL running
✅ pgvector package installed
✅ psycopg available
Database ibm_rag exists: True

✅ Step 4 completed


Step 5 — Verify the Restored IBM Database

In [ ]:
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"

print("STEP 5 — DATABASE VERIFICATION")
print("=" * 45)

with psycopg.connect(DATABASE_URL) as conn:

    documents = conn.execute(
        "SELECT COUNT(*) FROM documents"
    ).fetchone()[0]

    semantic_chunks = conn.execute(
        "SELECT COUNT(*) FROM semantic_chunks"
    ).fetchone()[0]

    retrieval_chunks = conn.execute(
        "SELECT COUNT(*) FROM retrieval_chunks_overlap_v1"
    ).fetchone()[0]

    embedding_table = conn.execute(
        "SELECT to_regclass('public.chunk_embeddings')"
    ).fetchone()[0]

    vector_version = conn.execute("""
        SELECT extversion
        FROM pg_extension
        WHERE extname = 'vector'
    """).fetchone()

print("IBM documents:", documents)
print("Semantic chunks:", semantic_chunks)
print("Retrieval chunks:", retrieval_chunks)
print("Embedding table exists:", embedding_table is not None)
print(
    "pgvector extension:",
    vector_version[0] if vector_version else "Not enabled"
)

assert documents == 496
assert semantic_chunks == 3200
assert retrieval_chunks == 5098
assert embedding_table is None, "Old embedding table detected!"

print("\n✅ Chunking database verified")
print("✅ Ready to create fresh embeddings")

STEP 5 — DATABASE VERIFICATION
IBM documents: 496
Semantic chunks: 3200
Retrieval chunks: 5098
Embedding table exists: False
pgvector extension: Not enabled

✅ Chunking database verified
✅ Ready to create fresh embeddings


Step 6 — Enable pgvector and Create Embedding Table

In [ ]:
import subprocess
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"

print("STEP 6 — EMBEDDING TABLE SETUP")
print("=" * 45)

# 1. Enable pgvector in the restored database
subprocess.run(
    [
        "runuser", "-u", "postgres", "--",
        "psql", "-X", "-v", "ON_ERROR_STOP=1",
        "-d", "ibm_rag",
        "-c", "CREATE EXTENSION IF NOT EXISTS vector;"
    ],
    check=True,
    capture_output=True,
    text=True
)

# 2. Create the embedding table
with psycopg.connect(DATABASE_URL) as conn:

    conn.execute("""
        CREATE TABLE IF NOT EXISTS chunk_embeddings (
            optimized_id UUID NOT NULL
                REFERENCES retrieval_chunks_overlap_v1(optimized_id)
                ON DELETE CASCADE,

            model_name TEXT NOT NULL,
            embedding VECTOR(1024) NOT NULL,

            content_sha256 TEXT NOT NULL
                CHECK (LENGTH(content_sha256) = 64),

            created_at TIMESTAMPTZ DEFAULT NOW(),
            updated_at TIMESTAMPTZ DEFAULT NOW(),

            PRIMARY KEY (optimized_id, model_name)
        )
    """)

    # 3. Verify the result
    version = conn.execute("""
        SELECT extversion
        FROM pg_extension
        WHERE extname = 'vector'
    """).fetchone()[0]

    dimensions = conn.execute("""
        SELECT atttypmod
        FROM pg_attribute
        WHERE attrelid = 'chunk_embeddings'::regclass
          AND attname = 'embedding'
    """).fetchone()[0]

    total_chunks = conn.execute("""
        SELECT COUNT(*)
        FROM retrieval_chunks_overlap_v1
    """).fetchone()[0]

    saved = conn.execute("""
        SELECT COUNT(*)
        FROM chunk_embeddings
        WHERE model_name = %s
    """, (MODEL,)).fetchone()[0]

assert dimensions == 1024
assert total_chunks == 5098
assert saved == 0, "Expected an empty embedding table"

print("✅ pgvector enabled:", version)
print("✅ Embedding table created")
print("Vector dimensions:", dimensions)
print("Total retrieval chunks:", total_chunks)
print("Saved embeddings:", saved)
print("Remaining:", total_chunks - saved)

print("\n✅ Step 6 completed")

STEP 6 — EMBEDDING TABLE SETUP
✅ pgvector enabled: 0.6.0
✅ Embedding table created
Vector dimensions: 1024
Total retrieval chunks: 5098
Saved embeddings: 0
Remaining: 5098

✅ Step 6 completed


Step 7 — Test Embedding Generation

In [ ]:
import psycopg
import urllib.request
import json
import hashlib
import math
import time

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"
BATCH_SIZE = 10

print("STEP 7 — EMBEDDING TEST")
print("=" * 45)

# 1. Select chunks without embeddings
with psycopg.connect(DATABASE_URL) as conn:
    chunks = conn.execute("""
        SELECT c.optimized_id, c.content
        FROM retrieval_chunks_overlap_v1 c
        WHERE NOT EXISTS (
            SELECT 1
            FROM chunk_embeddings e
            WHERE e.optimized_id = c.optimized_id
              AND e.model_name = %s
        )
        ORDER BY c.document_id, c.chunk_index
        LIMIT %s
    """, (MODEL, BATCH_SIZE)).fetchall()

print("Selected chunks:", len(chunks))

if chunks:

    # 2. Generate embeddings with Qwen
    payload = json.dumps({
        "model": MODEL,
        "input": [content for _, content in chunks],
        "truncate": False
    }).encode("utf-8")

    request = urllib.request.Request(
        "http://127.0.0.1:11434/api/embed",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    start = time.perf_counter()

    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.load(response)

    vectors = result["embeddings"]
    elapsed = time.perf_counter() - start

    # 3. Validate vectors
    assert len(vectors) == len(chunks)

    records = []

    for (chunk_id, content), vector in zip(chunks, vectors):
        assert len(vector) == 1024
        assert all(math.isfinite(x) for x in vector)

        records.append((
            chunk_id,
            MODEL,
            json.dumps(vector),
            hashlib.sha256(content.encode("utf-8")).hexdigest()
        ))

    # 4. Save embeddings to PostgreSQL
    with psycopg.connect(DATABASE_URL) as conn:
        with conn.cursor() as cur:
            cur.executemany("""
                INSERT INTO chunk_embeddings (
                    optimized_id,
                    model_name,
                    embedding,
                    content_sha256
                )
                VALUES (%s, %s, %s::vector, %s)
                ON CONFLICT (optimized_id, model_name)
                DO NOTHING
            """, records)

    print("✅ Embeddings generated and saved")
    print("Processing time:", round(elapsed, 2), "seconds")

# 5. Verify saved embeddings
with psycopg.connect(DATABASE_URL) as conn:
    saved = conn.execute("""
        SELECT COUNT(*)
        FROM chunk_embeddings
        WHERE model_name = %s
    """, (MODEL,)).fetchone()[0]

print("\nDATABASE STATUS")
print("Total chunks:", 5098)
print("Saved embeddings:", saved)
print("Remaining:", 5098 - saved)

print("\n✅ Step 7 completed")

STEP 7 — EMBEDDING TEST
Selected chunks: 10
✅ Embeddings generated and saved
Processing time: 20.95 seconds

DATABASE STATUS
Total chunks: 5098
Saved embeddings: 10
Remaining: 5088

✅ Step 7 completed


Step 8 — Full Embedding Generation with Automatic Checkpoints

In [ ]:
import psycopg
import urllib.request
import json
import hashlib
import math
import time
import subprocess
import shutil
import os
from pathlib import Path

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"
OLLAMA_URL = "http://127.0.0.1:11434"

BATCH_SIZE = 32
CHECKPOINT_EVERY = 256

BACKUP_DIR = Path(
    "/content/drive/MyDrive/IBM_GraphRAG/postgres_backups"
)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

BACKUP_PATH = BACKUP_DIR / "ibm_embeddings_progress.dump"
LOCAL_BACKUP = Path("/content/ibm_embeddings_progress.dump")

print("STEP 8 — FULL EMBEDDING GENERATION")
print("=" * 50)


def get_counts():
    with psycopg.connect(DATABASE_URL) as conn:
        total = conn.execute(
            "SELECT COUNT(*) FROM retrieval_chunks_overlap_v1"
        ).fetchone()[0]

        saved = conn.execute("""
            SELECT COUNT(*)
            FROM chunk_embeddings
            WHERE model_name = %s
        """, (MODEL,)).fetchone()[0]

    return total, saved


def save_checkpoint():
    print("\nSaving PostgreSQL checkpoint...")

    subprocess.run([
        "pg_dump", "-Fc",
        "--dbname", DATABASE_URL,
        "--file", str(LOCAL_BACKUP)
    ], check=True)

    temp_path = Path(str(BACKUP_PATH) + ".tmp")
    shutil.copy2(LOCAL_BACKUP, temp_path)

    subprocess.run([
        "pg_restore", "--list", str(temp_path)
    ], check=True, stdout=subprocess.DEVNULL)

    os.replace(temp_path, BACKUP_PATH)

    print("✅ Checkpoint saved to Google Drive")


total, saved = get_counts()

print("Total chunks:", total)
print("Already embedded:", saved)
print("Remaining:", total - saved)

assert total == 5098, "Unexpected chunk count"

processed = 0
since_checkpoint = 0
last_checkpoint = time.monotonic()
start = time.monotonic()

try:
    while True:

        # Select only chunks without embeddings
        with psycopg.connect(DATABASE_URL) as conn:
            chunks = conn.execute("""
                SELECT c.optimized_id, c.content
                FROM retrieval_chunks_overlap_v1 c
                WHERE NOT EXISTS (
                    SELECT 1
                    FROM chunk_embeddings e
                    WHERE e.optimized_id = c.optimized_id
                      AND e.model_name = %s
                )
                ORDER BY c.document_id, c.chunk_index
                LIMIT %s
            """, (MODEL, BATCH_SIZE)).fetchall()

        if not chunks:
            break

        # Generate embeddings
        payload = json.dumps({
            "model": MODEL,
            "input": [content for _, content in chunks],
            "truncate": False,
            "keep_alive": "15m"
        }).encode("utf-8")

        request = urllib.request.Request(
            f"{OLLAMA_URL}/api/embed",
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST"
        )

        with urllib.request.urlopen(request, timeout=600) as response:
            vectors = json.load(response)["embeddings"]

        assert len(vectors) == len(chunks)

        records = []

        for (chunk_id, content), vector in zip(chunks, vectors):
            assert len(vector) == 1024
            assert all(math.isfinite(x) for x in vector)

            records.append((
                chunk_id,
                MODEL,
                json.dumps(vector),
                hashlib.sha256(content.encode("utf-8")).hexdigest()
            ))

        # Save batch to PostgreSQL
        with psycopg.connect(DATABASE_URL) as conn:
            with conn.cursor() as cur:
                cur.executemany("""
                    INSERT INTO chunk_embeddings
                        (optimized_id, model_name, embedding, content_sha256)
                    VALUES (%s, %s, %s::vector, %s)
                    ON CONFLICT (optimized_id, model_name) DO NOTHING
                """, records)

        processed += len(chunks)
        since_checkpoint += len(chunks)

        total, saved = get_counts()

        print(
            f"✅ Embeddings: {saved}/{total} "
            f"| Remaining: {total - saved}",
            flush=True
        )

        # Checkpoint every 256 chunks or 2 minutes
        if (
            since_checkpoint >= CHECKPOINT_EVERY
            or time.monotonic() - last_checkpoint >= 120
        ):
            save_checkpoint()
            since_checkpoint = 0
            last_checkpoint = time.monotonic()

finally:
    # Preserve committed progress even after an interruption
    if since_checkpoint > 0 or not BACKUP_PATH.exists():
        save_checkpoint()

total, saved = get_counts()

print("\nFINAL RESULT")
print("=" * 50)
print("Total chunks:", total)
print("Saved embeddings:", saved)
print("Remaining:", total - saved)
print("Processed this run:", processed)
print("Runtime (minutes):", round((time.monotonic() - start) / 60, 2))
print("Backup:", BACKUP_PATH)

if saved == total:
    print("\n✅ ALL EMBEDDINGS COMPLETED")
else:
    print("\n⚠️ Some embeddings are missing")

STEP 8 — FULL EMBEDDING GENERATION
Total chunks: 5098
Already embedded: 10
Remaining: 5088
✅ Embeddings: 42/5098 | Remaining: 5056
✅ Embeddings: 74/5098 | Remaining: 5024
✅ Embeddings: 106/5098 | Remaining: 4992
✅ Embeddings: 138/5098 | Remaining: 4960
✅ Embeddings: 170/5098 | Remaining: 4928
✅ Embeddings: 202/5098 | Remaining: 4896
✅ Embeddings: 234/5098 | Remaining: 4864
✅ Embeddings: 266/5098 | Remaining: 4832

Saving PostgreSQL checkpoint...
✅ Checkpoint saved to Google Drive
✅ Embeddings: 298/5098 | Remaining: 4800
✅ Embeddings: 330/5098 | Remaining: 4768
✅ Embeddings: 362/5098 | Remaining: 4736
✅ Embeddings: 394/5098 | Remaining: 4704
✅ Embeddings: 426/5098 | Remaining: 4672
✅ Embeddings: 458/5098 | Remaining: 4640
✅ Embeddings: 490/5098 | Remaining: 4608
✅ Embeddings: 522/5098 | Remaining: 4576

Saving PostgreSQL checkpoint...
✅ Checkpoint saved to Google Drive
✅ Embeddings: 554/5098 | Remaining: 4544
✅ Embeddings: 586/5098 | Remaining: 4512
✅ Embeddings: 618/5098 | Remaining: 4

Step 9 — Create HNSW Index and Final Database Backup

In [ ]:
import psycopg
import subprocess
import shutil
import os
from pathlib import Path

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"

BACKUP_DIR = Path(
    "/content/drive/MyDrive/IBM_GraphRAG/postgres_backups"
)
FINAL_BACKUP = BACKUP_DIR / "ibm_embeddings_final.dump"
LOCAL_BACKUP = Path("/content/ibm_embeddings_final.dump")

print("STEP 9 — HNSW INDEX + FINAL BACKUP")
print("=" * 50)

# 1. Verify all embeddings are present
with psycopg.connect(DATABASE_URL) as conn:
    total = conn.execute(
        "SELECT COUNT(*) FROM retrieval_chunks_overlap_v1"
    ).fetchone()[0]

    saved = conn.execute("""
        SELECT COUNT(*)
        FROM chunk_embeddings
        WHERE model_name = %s
    """, (MODEL,)).fetchone()[0]

assert total == saved == 5098, "Embeddings are incomplete!"

print(f"✅ Embeddings verified: {saved}/{total}")

# 2. Create HNSW index for cosine similarity
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute("""
        CREATE INDEX IF NOT EXISTS chunk_embeddings_qwen_hnsw_idx
        ON chunk_embeddings
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64)
        WHERE model_name = 'qwen3-embedding:0.6b'
    """)

    conn.execute("ANALYZE chunk_embeddings")

    index_valid = conn.execute("""
        SELECT i.indisvalid
        FROM pg_index i
        JOIN pg_class c ON c.oid = i.indexrelid
        WHERE c.relname = 'chunk_embeddings_qwen_hnsw_idx'
    """).fetchone()

assert index_valid and index_valid[0], "HNSW index is not valid"

print("✅ HNSW index created and verified")

# 3. Back up the complete database
subprocess.run([
    "pg_dump", "-Fc",
    "--dbname", DATABASE_URL,
    "--file", str(LOCAL_BACKUP)
], check=True)

# 4. Copy safely to Google Drive
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
temporary = Path(str(FINAL_BACKUP) + ".tmp")

shutil.copy2(LOCAL_BACKUP, temporary)

# Check that the archive includes embeddings and index
result = subprocess.run(
    ["pg_restore", "--list", str(temporary)],
    capture_output=True,
    text=True,
    check=True
)

assert "chunk_embeddings" in result.stdout
assert "chunk_embeddings_qwen_hnsw_idx" in result.stdout

os.replace(temporary, FINAL_BACKUP)

print("✅ Final backup saved")
print("Backup:", FINAL_BACKUP)
print("Size:", round(FINAL_BACKUP.stat().st_size / 1024**2, 2), "MB")
print("\n✅ EMBEDDING NOTEBOOK COMPLETED")

STEP 9 — HNSW INDEX + FINAL BACKUP
✅ Embeddings verified: 5098/5098
✅ HNSW index created and verified
✅ Final backup saved
Backup: /content/drive/MyDrive/IBM_GraphRAG/postgres_backups/ibm_embeddings_final.dump
Size: 28.34 MB

✅ EMBEDDING NOTEBOOK COMPLETED


### Step 10 — Test HNSW Nearest Neighbor Search

In [ ]:
import json
import urllib.request
import psycopg

DATABASE_URL = "postgresql:///ibm_rag?user=root"
MODEL = "qwen3-embedding:0.6b"

QUESTION = (
    "Where can I find the license terms and legal notices "
    "for third-party software included in an IBM product?"
)

print("QUESTION:")
print(QUESTION)

# 1. Convert the question into a 1024-dimensional vector
payload = json.dumps({
    "model": MODEL,
    "input": QUESTION
}).encode("utf-8")

request = urllib.request.Request(
    "http://127.0.0.1:11434/api/embed",
    data=payload,
    headers={"Content-Type": "application/json"},
    method="POST"
)

with urllib.request.urlopen(request, timeout=300) as response:
    result = json.load(response)

query_vector = json.dumps(result["embeddings"][0])

# 2. Find the 3 closest chunks using HNSW
with psycopg.connect(DATABASE_URL) as conn:
    conn.execute("SET LOCAL hnsw.ef_search = 100")

    results = conn.execute("""
        SELECT
            d.title,
            c.document_id,
            c.optimized_id,
            c.content,
            1 - (e.embedding <=> %s::vector) AS similarity

        FROM chunk_embeddings e
        JOIN retrieval_chunks_overlap_v1 c
            ON c.optimized_id = e.optimized_id
        JOIN documents d
            ON d.document_id = c.document_id

        WHERE e.model_name = %s

        ORDER BY e.embedding <=> %s::vector
        LIMIT 3
    """, (query_vector, MODEL, query_vector)).fetchall()

# 3. Display the results
print("\nTOP 3 CLOSEST CHUNKS")
print("=" * 60)

for rank, (title, doc_id, chunk_id, content, similarity) in enumerate(results, 1):
    print(f"\nRANK {rank}")
    print("Document:", title)
    print("Document ID:", doc_id)
    print("Chunk ID:", chunk_id)
    print("Cosine similarity:", round(similarity, 4))
    print("Text:")
    print(content[:600])
    print("-" * 60)

QUESTION:
Where can I find the license terms and legal notices for third-party software included in an IBM product?

TOP 3 CLOSEST CHUNKS

RANK 1
Document: IBM Tivoli Business Service Manager V6.1.1 Fix Pack 3(6.1.1-TIV-BSM-FP0003) - United States
Document ID: af0f8b8cae543c4c62db26f0be61bd63ee34b4f93db02093508f8d97a4e32399
Chunk ID: 22d530f7-aabb-5e5f-a79b-5936ee4145f6
Cosine similarity: 0.7883
Text:
8.2 THIRD-PARTY LICENSE TERMS AND CONDITIONS, NOTICES AND INFORMATION

The license agreement for this product refers you to this file for
details concerning terms and conditions applicable to third party
software code included in this product, and for certain notices and
other information IBM must provide to you under its license to certain
software code. The relevant terms and conditions, notices and other
information are provided or referenced below. Please note that any
non-English version of the licenses below is unofficial and is provided
to you for your convenience only. The English